# Cohort — Surgery and Fitbit

Defines the analysis cohort every other notebook reads: participants with Fitbit data who have a
procedure that is a descendant of OMOP concept `4301351` (SNOMED 387713003, "Surgical
procedure"), so the cohort spans all surgery rather than any one specialty. The definition is
derived here from the CDR's own vocabulary tables via `srp_common.cohort_ctes`, **not** read from
the Workbench cohort-builder index, so it lives in version-controlled code and cannot be narrowed
by UI state. A separate EHR test is not needed: `procedure_occurrence` rows are EHR data, so
requiring a qualifying procedure already implies it.

**Governance.** Outputs are stripped on commit, so anything rendered here stays local. Even so,
prefer aggregates over record listings — see the `allofus-data-governance` skill. Figures belong
in `surgery_fitbit_figures/` and query results in `_cache/`, both git-ignored.

## Shared Imports and Methods

Locates the repo root by walking up from the working directory, resolves the pinned CDR, creates
this notebook's figure and cache directories, and declares every column the notebook reads.
**`require_columns` is what turns a renamed or missing column into a failure here rather than a
BigQuery 400 several minutes into a run**, and `cohort_ctes` validates its own tables separately.

#### Imports, CDR Resolution and the Column Contract

Everything the rest of the notebook assumes about its environment is settled in this one cell, so
a run that gets past it cannot fail on setup later. The CDR is pinned in `cdr_config.py` rather
than auto-selected, because a newer release appearing underneath a finished analysis would shift
its results silently.

In [ ]:
import sys
from pathlib import Path

import pandas

# Locate the repo root -- the directory holding srp_common.py -- by walking up from
# cwd, so this works whether the kernel starts in the task directory or the repo root.
_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common

# Resolves the pinned CDR and creates this notebook's git-ignored figure and cache
# directories. Every query below reads the CDR only.
cfg = srp_common.init("surgery_fitbit")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Fail here, not several minutes into a BigQuery run, if the CDR does not have
# what this notebook reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "concept": ["concept_id", "concept_name", "vocabulary_id", "concept_code"],
    "person": ["person_id", "birth_datetime", "gender_concept_id", "sex_at_birth_concept_id"],
    "procedure_occurrence": ["procedure_type_concept_id", "procedure_source_concept_id", "modifier_source_value", "visit_occurrence_id"],
})

## Part 1 — Is This Cohort What It Is Assumed To Be

Two things have to hold before anything downstream means what it says. **The concept set has to
be the whole of surgery rather than a fragment of it** — descendants of "Surgical procedure"
number in the tens of thousands, so a set in the low hundreds would mean the ancestor is not what
it is taken to be — and the tables the cohort carries have to have the columns and the row counts
the later notebooks expect. Both are read as aggregates here; the record listings that the
generated cohort-builder code ends with are deliberately absent.

#### Breadth, Size, and What the Two Extracts Hold

The concept count settles the cohort's breadth from data rather than from the ancestor's name,
and the three participant counts show how much is lost at each narrowing — Fitbit, then a
qualifying procedure, then a dated one. **The person and procedure extracts that follow are
reported as shape and schema only, never as rows**, because a listing that lands in saved output
is a DUA problem rather than a style one.

In [ ]:
# How broad is the surgical concept set, and how big is the cohort it selects?
# This is the check that settles the cohort's breadth from data rather than from a name:
# descendants of "Surgical procedure" number in the tens of thousands, so a set in the low
# hundreds would mean the ancestor is not what it is assumed to be.

cohort_size_sql = cfg.cohort_ctes() + """
    SELECT
        (SELECT COUNT(*) FROM surgical_concept) AS n_surgical_concepts,
        (SELECT COUNT(*) FROM fitbit_person)    AS n_fitbit_participants,
        (SELECT COUNT(*) FROM cohort)           AS n_cohort_patients,
        (SELECT COUNT(*) FROM index_surgery)    AS n_with_dated_surgery
"""

cohort_size = cached_gbq(cohort_size_sql, "cohort_size").iloc[0]

print(f"concepts in the surgical set : {int(cohort_size['n_surgical_concepts']):>9,}")
print(f"participants with Fitbit data: {int(cohort_size['n_fitbit_participants']):>9,}")
print(f"cohort (Fitbit + surgery)    : {int(cohort_size['n_cohort_patients']):>9,}")
print(f"  ... with a dated surgery   : {int(cohort_size['n_with_dated_surgery']):>9,}")

# The two tables the cohort carries. Both restrict through the `cohort` CTE, so
# neither can drift from the definition checked immediately above.
person_sql = cfg.cohort_ctes() + f"""
    SELECT
        p.person_id,
        p.birth_datetime,
        p.gender_concept_id,
        gender.concept_name      AS gender,
        p.sex_at_birth_concept_id,
        sex.concept_name         AS sex_at_birth
    FROM `{CDR}`.person AS p
    JOIN cohort AS c ON c.person_id = p.person_id
    LEFT JOIN `{CDR}`.concept AS gender ON gender.concept_id = p.gender_concept_id
    LEFT JOIN `{CDR}`.concept AS sex    ON sex.concept_id    = p.sex_at_birth_concept_id
"""

person_df = cached_gbq(person_sql, "person")

procedure_occurrence_sql = cfg.cohort_ctes() + f"""
    SELECT
        po.person_id,
        po.procedure_concept_id,
        po.procedure_datetime,
        po.procedure_type_concept_id,
        ptype.concept_name       AS procedure_type,
        po.procedure_source_concept_id,
        source.vocabulary_id     AS source_vocabulary,
        standard.concept_code    AS standard_concept_code,
        standard.concept_name    AS standard_concept_name,
        standard.vocabulary_id   AS standard_vocabulary,
        po.modifier_source_value,
        po.visit_occurrence_id
    FROM `{CDR}`.procedure_occurrence AS po
    JOIN cohort AS c ON c.person_id = po.person_id
    LEFT JOIN `{CDR}`.concept AS ptype    ON ptype.concept_id    = po.procedure_type_concept_id
    LEFT JOIN `{CDR}`.concept AS source   ON source.concept_id   = po.procedure_source_concept_id
    LEFT JOIN `{CDR}`.concept AS standard ON standard.concept_id = po.procedure_concept_id
    WHERE po.procedure_concept_id IN (SELECT concept_id FROM surgical_concept)
"""

procedure_occurrence_df = cached_gbq(procedure_occurrence_sql, "procedure_occurrence")

# Aggregate summary only — no participant-level record listing.
# The original notebook ended each query with .head(5); eyeballing rows interactively is
# fine, but a listing that lands in saved output is a DUA problem, so the default here is
# shape + schema. Suppress small cells before reporting any count table externally.

for label, df in [("person", person_df), ("procedure_occurrence", procedure_occurrence_df)]:
    print(f"{label:22s} rows={len(df):>9,}  cols={df.shape[1]:>3}  "
          f"unique person_id={df['person_id'].nunique():>8,}")

print()
print("person columns             :", list(person_df.columns))
print("procedure_occurrence columns:", list(procedure_occurrence_df.columns))